In [ ]:
"""Load and analyze failure reports. Cell 1: Load latest failures."""

import json
from pathlib import Path
from glob import glob

OUTPUT_DIR = Path("output")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("tests/output")

files = sorted(glob(str(OUTPUT_DIR / "failures_*.json")), reverse=True)
if not files:
    raise FileNotFoundError(f"No failure reports in {OUTPUT_DIR}")

LATEST = Path(files[0])
print(f"Loaded: {LATEST.name}")

with open(LATEST) as f:
    failures = json.load(f)

test_name = LATEST.stem.replace("failures_", "").rsplit("_", 1)[0]
print(f"Test: {test_name}")

In [ ]:
"""Cell 2: Summary table."""

import pandas as pd

dfs = {}
total_failures = 0

for cat, items in failures.items():
    if not items:
        continue
    df = pd.DataFrame(items)
    
    # Flatten case dict into columns
    case_df = pd.json_normalize(df["case"]).add_prefix("case_")
    
    # Flatten result dict into columns
    result_df = pd.json_normalize(df["result"]).add_prefix("result_")
    
    # Combine
    flat_df = pd.concat([df[["index"]], case_df, result_df], axis=1)
    flat_df["category"] = cat
    flat_df["test"] = test_name
    
    dfs[cat] = flat_df
    total_failures += len(flat_df)

# Summary table
summary = pd.DataFrame([
    {"category": cat, "failures": len(df), "% of total": f"{len(df)/total_failures*100:.1f}%"}
    for cat, df in dfs.items()
])
summary = summary.sort_values("failures", ascending=False).reset_index(drop=True)
print(f"\n=== SUMMARY: {total_failures} total failures ===\n")
print(summary.to_string(index=False))

In [ ]:
"""Cell 3: Show first few failures per category (flattened, easy to read)."""

for cat, df in dfs.items():
    print(f"\n=== {cat.upper()} (first 5) ===")
    # Show key columns
    cols = [c for c in df.columns if c.startswith("result_")]
    display(df[cols].head())

In [ ]:
"""Cell 4: Filter by ending type."""

# Example: show failures grouped by ending
all_df = pd.concat(dfs.values(), ignore_index=True)
print("Failures by ending:")
print(all_df.groupby("result_ending").size().sort_values(ascending=False))

In [ ]:
"""Cell 5: Filter by status."""

print("Failures by status:")
print(all_df.groupby("result_status").size().sort_values(ascending=False))

In [ ]:
"""Cell 6: Drill down into specific category/ending combo."""

# Example: show ordinary failures with 'radd' ending
radd_failures = all_df[all_df["result_ending"] == "radd"]
print(f"Radd failures: {len(radd_failures)}")
display(radd_failures.head(3))

In [ ]:
"""Cell 7: What heirs are present in failing cases?"""

case_cols = [c for c in all_df.columns if c.startswith("case_")]
heir_presence = all_df[case_cols].apply(lambda r: r > 0).sum()
print("Heir presence in failing cases:")
print(heir_presence.sort_values(ascending=False))

In [ ]:
"""Cell 8: Peek at raw case/result for any row."""

# Usage: peek(dfs['ordinary'], 5)
def peek(df, idx):
    row = df.iloc[idx]
    print(f"Index: {row['index']}")
    print(f"Category: {row['category']}")
    print("\nCase:")
    for col in case_cols:
        if row[col]:
            print(f"  {col.replace('case_', '')}: {row[col]}")
    print("\nResult:")
    print(f"  ending: {row['result_ending']}")
    print(f"  status: {row['result_status']}")
    print(f"  total: {row['result_total']}")
    print(f"  distribution: {row['result_distribution']}")

# Example: peek(dfs['ordinary'], 0)
# peek(dfs['ordinary'], 0)